In [1]:
from jaxcmr.memorysearch import free_recall, trial_item_count, recall_by_item_index
from jaxcmr.evaluation import extract_objective_data
from jaxcmr.datasets import load_data, generate_trial_mask, load_parameters
from jaxcmr.analyses import single_pres_spc, single_pres_pfr, single_pres_crp
import numpy as np

import jax
from jax import numpy as jnp

from scipy.optimize import differential_evolution
import json
from tqdm import tqdm
import os

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [2]:
def fit_model_to_subject(
        data, loss_fn, model_create_fn, fixed_parameters, free_parameters, bounds, subject,  trial_count, model_name, 
        trial_mask, embeddings, ignore_first_recall=False, display=False):
    """Fit a model to a given subject in a dataset."""

    if trial_mask is None:
        trial_mask = jnp.ones(len(data['recalls']), dtype=np.bool_)

    presentations = jnp.array(data['pres_itemnos'][trial_mask])
    trials = jax.vmap(recall_by_item_index)(
        presentations, jnp.array(data['recalls'][trial_mask]))

    base_parameters = fixed_parameters.copy()
    loss_fn = loss_fn(
        model_create_fn, 
        presentations, 
        trials
        )

    @jax.jit
    def objective_function(x):
        for key_index, key in enumerate(free_parameters):
            base_parameters[key] = x[key_index]
        return loss_fn(base_parameters)

    fit_result = differential_evolution(objective_function, bounds, disp=display, tol=.001)
    fitted_parameters = {
        'subject': int(subject), 'trial_count': int(trial_count),
        'likelihood': float(fit_result.fun), 'model': str(model_name), 'fixed': {}, 'free': {}
        }

    for key in fixed_parameters:
        fitted_parameters['fixed'][key] = float(fixed_parameters[key])
    for key_index, key in enumerate(free_parameters):
        fitted_parameters['fixed'][key] = float(fit_result.x[key_index])
    for key in free_parameters:
        fitted_parameters['free'][key] = list(free_parameters[key])

    return fitted_parameters

In [3]:
def fit_model_to_dataset(
        loss_fn, # loss function to use for fitting
        model_create_fn, # function to use for creating the model
        fit_fn, # function to use for fitting
        data_path, # path to the dataset
        parameter_path, # path to parameter dictionary specifying fixed and free parameters
        fit_by_subject=True, # whether to fit by subject
        ignore_first_recall=False, # whether to ignore the first recall in each list
        embedding_paths=None, # a list of paths to embedding files to pass to the model
        trial_query=None, # a query to select trials to fit
        model_name=None, # name of the model
        result_path=None, # path to JSONL for saving, retrieving results
        replace_results=False, # whether to replace existing results
        display=False, # whether to display fitting progress
        ):
    "Fit a model to a dataset based on the given parameters."

    # load data and parameters
    data = load_data(data_path)
    parameters = load_parameters(parameter_path)
    
    trial_mask = None
    if trial_query is not None:
        trial_mask = generate_trial_mask(data, trial_query)

    embeddings = None
    if embedding_paths is not None:
        embeddings = load_embeddings(embedding_paths)

    # This uses the old parameter format; maybe should be updated to use the new one
    fixed_parameters = dict(parameters['fixed'])
    free_parameters = parameters['free']
    bounds = np.zeros((len(free_parameters), 2))
    for key_index, key in enumerate(free_parameters):
        bounds[key_index, 0] = free_parameters[key][0]
        bounds[key_index, 1] = free_parameters[key][1]

    subject_fits = []
    already_fit = []
    subject_indices = np.unique(data['subject']) if fit_by_subject else [-1]

    # if we are replacing results, let's create a temporary file to write to
    if replace_results:
        result_path = result_path + '_tmp'
        if os.path.exists(result_path):
            os.remove(result_path)

    # load pre-computed results and identify subjects that have already been fit
    if (not replace_results) and (result_path is not None) and (os.path.exists(result_path)):
        with open(result_path, 'r') as f:
            for line in f:
                subject_fits.append(json.loads(line))
        already_fit = [subject_fit['subject'] for subject_fit in subject_fits]

    index_loop = tqdm(subject_indices)
    for subject in index_loop:

        if subject in already_fit:
            continue

        # need subject-specific trial mask if fitting by subject (index != -1)
        if subject >= 0:
            if trial_mask is None:
                trial_mask = np.ones(len(data['recalls']), dtype=bool)
            subject_specific_trial_mask = np.logical_and(trial_mask, (data['subject'] == subject).flatten())
        else:
            subject_specific_trial_mask = trial_mask.copy()

        # skip subjects with no trials to fit model to
        trial_count = np.sum(subject_specific_trial_mask)
        if trial_count == 0:
            continue

        if display:
            print("Fitting subject {} with {} trials".format(subject, trial_count))

        fitted_parameters = fit_fn(
            data, loss_fn, model_create_fn, fixed_parameters, free_parameters, bounds, subject, trial_count, model_name, 
            subject_specific_trial_mask, embeddings, ignore_first_recall, display)

        subject_fits.append(fitted_parameters)
        if result_path is not None:
            with open(result_path, 'a') as f:
                f.write(json.dumps(fitted_parameters) + '\n')

        index_loop.set_description("Last fit: {}".format(fitted_parameters['likelihood']))

    if len(subject_fits) == 0:
        raise ValueError('No subjects to fit.')

    # if we are replacing results, let's replace the old file with the new one
    if replace_results:
        if os.path.exists(result_path[:-4]):
            os.remove(result_path[:-4])
        os.rename(result_path, result_path[:-4])

    return subject_fits

# LohnasKahana2014

In [5]:
from jaxcmr.memorysearch import BaseCMR, InstanceCMR, variable_presentations_data_likelihood

model_name = 'Base_CMR'
parameter_path = 'D:/data/base_cmr_parameters.json'

data_tag = 'LohnasKahana2014'
trial_query = "data['list_type'] != -1"

result_path = 'D:/data/results/{}_{}_{}.jsonl'
data_path = 'D:/data/{}.h5'

ignore_first_recall = False

fit_model_to_dataset(
    variable_presentations_data_likelihood,
    InstanceCMR.create,
    fit_model_to_subject,
    data_path.format(data_tag), # path to the dataset
    parameter_path, # path to Parameter object specifying fixed and free parameters for fitting
    fit_by_subject = True, # whether to fit by subject
    ignore_first_recall=ignore_first_recall, # whether to ignore the first recall in each list
    embedding_paths=None, # a list of paths to embedding files to pass to the model
    trial_query=trial_query, # a query to select trials to fit
    model_name=model_name, # name of the model
    result_path=result_path.format(model_name, data_tag, ignore_first_recall), # path to JSONL for saving, retrieving results
    replace_results=False, # whether to replace existing results
    display=False, # whether to display fitting progress
    );

  0%|          | 0/35 [00:00<?, ?it/s]

d:\projects\jaxcmr\venv\Lib\site-packages\beartype\_util\hint\pep\utilpeptest.py:311: BeartypeDecorHintPep585DeprecationWarning: PEP 484 type hint typing.Callable deprecated by PEP 585. This hint is scheduled for removal in the first Python version released after October 5th, 2025. To resolve this, import this hint from "beartype.typing" rather than "typing". For further commentary and alternatives, see also:
    https://beartype.readthedocs.io/en/latest/api_roar/#pep-585-deprecations
  warn(
Last fit: 1722.256103515625:  46%|████▌     | 16/35 [44:25<46:51, 148.00s/it]  d:\projects\jaxcmr\venv\Lib\site-packages\scipy\optimize\_numdiff.py:576: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
Last fit: 880.9038696289062: 100%|██████████| 35/35 [1:37:22<00:00, 166.94s/it] 


In [6]:
model_name = 'Base_ICMR'
parameter_path = 'D:/data/base_cmr_parameters.json'

result_path = 'D:/data/results/{}_{}_{}.jsonl'
data_path = 'D:/data/{}.h5'

ignore_first_recall = False

fit_model_to_dataset(
    variable_presentations_data_likelihood,
    InstanceCMR.create,
    fit_model_to_subject,
    data_path.format(data_tag), # path to the dataset
    parameter_path, # path to Parameter object specifying fixed and free parameters for fitting
    fit_by_subject = True, # whether to fit by subject
    ignore_first_recall=ignore_first_recall, # whether to ignore the first recall in each list
    embedding_paths=None, # a list of paths to embedding files to pass to the model
    trial_query=trial_query, # a query to select trials to fit
    model_name=model_name, # name of the model
    result_path=result_path.format(model_name, data_tag, ignore_first_recall), # path to JSONL for saving, retrieving results
    replace_results=False, # whether to replace existing results
    display=False, # whether to display fitting progress
    );

Last fit: 1100.8607177734375:  74%|███████▍  | 26/35 [1:19:13<24:45, 165.01s/it] 

In [ ]:
model_name = 'trace_ICMR'
parameter_path = 'D:/data/instance_cmr_parameters.json'

result_path = 'D:/data/results/{}_{}_{}.jsonl'
data_path = '../../data/{}.h5'

ignore_first_recall = False

fit_model_to_dataset(
    variable_presentations_data_likelihood,
    fit_model_to_subject,
    data_path.format(data_tag), # path to the dataset
    parameter_path, # path to Parameter object specifying fixed and free parameters for fitting
    fit_by_subject = True, # whether to fit by subject
    ignore_first_recall=ignore_first_recall, # whether to ignore the first recall in each list
    embedding_paths=None, # a list of paths to embedding files to pass to the model
    trial_query=trial_query, # a query to select trials to fit
    model_name=model_name, # name of the model
    result_path=result_path.format(model_name, data_tag, ignore_first_recall), # path to JSONL for saving, retrieving results
    replace_results=False, # whether to replace existing results
    display=False, # whether to display fitting progress
    );

Last fit: 657.2520141601562: 100%|██████████| 126/126 [36:18<00:00, 17.29s/it] 


In [ ]:
model_name = 'dual_ICMR'
parameter_path = 'D:/data/dual_cmr_parameters.json'

result_path = 'D:/data/results/{}_{}_{}.jsonl'
data_path = 'D:/data/{}.h5'

ignore_first_recall = False

fit_model_to_dataset(
    variable_presentations_data_likelihood,
    fit_model_to_subject,
    data_path.format(data_tag), # path to the dataset
    parameter_path, # path to Parameter object specifying fixed and free parameters for fitting
    fit_by_subject = True, # whether to fit by subject
    ignore_first_recall=ignore_first_recall, # whether to ignore the first recall in each list
    embedding_paths=None, # a list of paths to embedding files to pass to the model
    trial_query=trial_query, # a query to select trials to fit
    model_name=model_name, # name of the model
    result_path=result_path.format(model_name, data_tag, ignore_first_recall), # path to JSONL for saving, retrieving results
    replace_results=True, # whether to replace existing results
    display=False, # whether to display fitting progress
    );

TypeError: fit_model_to_dataset() missing 1 required positional argument: 'parameter_path'